# User Simulator Spot-Checking

Interactive one-message-at-a-time continuation from `items-sanitized.parquet`.

This notebook starts from the first `k` real messages of a selected window. It only allows prefixes where the next original message is a user turn. Qwen3-8B generates synthetic user messages; `gpt-4o-mini` generates synthetic assistant messages through the OpenAI API using `OPENAI_EXPT_API_KEY`.

Generated spot-checks are saved to `sim_outputs/spotchecks.jsonl` when you press **Save JSONL**.

## Setup

Run this notebook on the GPU machine. If dependencies are missing, install them in the active kernel environment before loading Qwen:

```bash
pip install torch transformers accelerate openai ipywidgets pandas pyarrow
# Optional for USE_4BIT=True:
pip install bitsandbytes
```

Set the assistant API key before starting Jupyter so the kernel can see it:

```bash
read -s -p "OPENAI_EXPT_API_KEY: " OPENAI_EXPT_API_KEY; echo
export OPENAI_EXPT_API_KEY
```

On Hyak, saved spot-checks default to `/gscratch/scrubbed/adhyyan/llm-delusions-evals/outputs/spotchecks.jsonl`. Override this with `SPOTCHECK_OUTPUT_PATH` before starting Jupyter, or edit the save path text box in the UI.

In [ ]:
from IPython.display import display
import pandas as pd

from user_simulator_spotcheck_helpers import *

pd.set_option('display.max_columns', 120)
pd.set_option('display.max_colwidth', 220)
pd.set_option('display.width', 180)

## Runtime Configuration

Confirm that the OpenAI key is visible to this kernel and that outputs are going to scratch on Hyak.


In [ ]:
display({
    'openai_api_key_env': OPENAI_API_KEY_ENV,
    'openai_api_key_visible_to_kernel': openai_api_key_is_set(),
    'output_path_env': OUTPUT_PATH_ENV,
    'default_output_path': str(default_output_path()),
})

if not openai_api_key_is_set():
    print('OPENAI_EXPT_API_KEY is not visible. If you already exported it, restart Jupyter from that same shell.')


## GPU And Dataset Checks

In [ ]:
display(check_gpu())

windows_df = load_windows()
display({
    'shape': windows_df.shape,
    'unique_windows': windows_df['eval_subset_id'].nunique(),
    'labels': windows_df['label'].nunique(),
    'min_messages': int(windows_df['message_count'].min()),
    'median_messages': float(windows_df['message_count'].median()),
    'max_messages': int(windows_df['message_count'].max()),
})

display(windows_df[['eval_subset_id', 'label', 'meets_code', 'message_count']].head())

## Eligible Prefixes

A prefix is eligible when `messages[k]` is a real user turn. The first generated message will then be a synthetic user turn.

In [ ]:
summary_rows = []
for k in [0, 1, 2, 3, 4, 5, 6, 8, 10, 12, 15]:
    summary_rows.append({'k': k, 'eligible_windows': len(eligible_prefixes(k, windows_df))})
display(pd.DataFrame(summary_rows))

display(eligible_prefixes(2, windows_df).head(10))

## Interactive Spot Checker

Use the controls below:

- **k**: number of real prefix messages to keep.
- **Reset**: reset the transcript to the selected real prefix.
- **Load Qwen**: load `Qwen/Qwen3-8B` on the GPU. Use 4-bit if memory is tight.
- **Generate next**: append exactly one synthetic message.
- **Fast forward N**: call the same one-message generation path `N` times.
- **Save JSONL**: append the current spot-check state to `sim_outputs/spotchecks.jsonl`.

In [ ]:
ui = build_spotcheck_ui(windows_df)

## Manual Helper Calls

The UI is the intended path, but these functions are available for direct debugging.

In [ ]:
# Example manual flow:
# eligible = eligible_prefixes(2, windows_df)
# reset_state(eligible.iloc[0]['eval_subset_id'], 2)
# load_qwen_model(use_4bit=False)
# generate_next()       # user message from Qwen
# generate_next()       # assistant message from gpt-4o-mini
# fast_forward(3)
# render_state()
# save_state_jsonl()

print('Available helpers: load_windows, eligible_prefixes, reset_state, generate_next, fast_forward, save_state_jsonl')

## Reload Saved Spot Checks

In [ ]:
saved = load_saved_rollouts()
print(f'Loaded {len(saved)} saved rollout record(s).')
if saved:
    last = saved[-1]
    display({k: last[k] for k in ['timestamp', 'eval_subset_id', 'source_label', 'k']})
    render_messages(
        last['real_prefix'],
        last['generated_messages'],
        title='Last saved spot-check',
        next_original_message=last.get('next_original_message'),
        metadata=last,
    )